# 🤖 Entrenamiento y Análisis del Modelo ML de Nutrición

Este notebook realiza:
1. ✅ Entrenamiento de nuevo modelo vía API
2. 📊 Obtención de métricas del último modelo
3. 📈 Generación de 6 gráficas clave del modelo

**Servidor**: `https://nutricion-modelo-ml-343042748851.us-east1.run.app`

## 📦 Instalación de Dependencias

In [ ]:
# Instalar librerías necesarias
!pip install requests matplotlib seaborn pandas numpy plotly -q

print("✅ Dependencias instaladas correctamente")

## 🔧 Configuración e Imports

In [ ]:
import requests
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# URL del servidor
BASE_URL = "https://nutricion-modelo-ml-343042748851.us-east1.run.app"

print("✅ Imports y configuración completados")
print(f"🌐 Servidor: {BASE_URL}")

## 🚀 PASO 1: Entrenar Nuevo Modelo

⚠️ **NOTA**: Este proceso toma varios minutos (2-5 min aprox.)

In [ ]:
# Entrenar nuevo modelo
print("🚀 Iniciando entrenamiento del modelo...")
print("⏳ Este proceso puede tomar 2-5 minutos...\n")

train_url = f"{BASE_URL}/api/v1/models/train/quick"

try:
    response = requests.post(train_url, timeout=600)
    
    if response.status_code == 200:
        result = response.json()
        
        print("✅ ENTRENAMIENTO COMPLETADO")
        print("="*60)
        print(f"📦 Modelo: {result.get('model_name', 'N/A')}")
        print(f"🕐 Timestamp: {result.get('timestamp', 'N/A')}")
        print(f"✅ Éxito: {result.get('success', False)}")
        print(f"💬 Mensaje: {result.get('message', 'N/A')}")
        print("="*60)
        
        if result.get('metrics'):
            print("\n📊 Métricas del nuevo modelo:")
            metrics = result['metrics']
            print(f"  • Accuracy Exacto: {metrics.get('accuracy_exact', 0):.2f}%")
            print(f"  • NDCG: {metrics.get('ndcg', 0):.2f}%")
            print(f"  • RMSE Val: {metrics.get('rmse_val', 0):.4f}")
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)
        
except requests.exceptions.Timeout:
    print("⏱️ El entrenamiento está tomando más tiempo del esperado.")
    print("Continúa con el siguiente paso para verificar el estado.")
except Exception as e:
    print(f"❌ Error durante el entrenamiento: {str(e)}")

## 📊 PASO 2: Obtener Métricas del Último Modelo

In [ ]:
# Obtener estado y métricas
print("📡 Obteniendo métricas del último modelo...\n")

status_url = f"{BASE_URL}/api/v1/models/train/status"

try:
    response = requests.get(status_url, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        
        if data['success'] and data['models_count'] > 0:
            # Obtener el último modelo
            models = data['models']
            latest_model = sorted(models, key=lambda x: x['created_at'], reverse=True)[0]
            
            # Guardar métricas para usar después
            metrics = latest_model['metrics']
            
            print("═"*70)
            print("📊 MÉTRICAS DEL ÚLTIMO MODELO ENTRENADO")
            print("═"*70)
            print(f"\n🆔 Modelo: {latest_model['name']}")
            print(f"📅 Fecha: {latest_model['created_at']}")
            print(f"💾 Tamaño: {latest_model['size_mb']} MB")
            print(f"📂 Path: {latest_model['path']}")
            
            print("\n" + "─"*70)
            print("📈 MÉTRICAS DE PRECISIÓN")
            print("─"*70)
            print(f"  ✓ Accuracy Exacto:        {metrics.get('accuracy_exact', 0):.2f}%")
            print(f"  ✓ Accuracy ±1:            {metrics.get('accuracy_tolerance_1', 0):.2f}%")
            print(f"  ✓ R² Score:               {metrics.get('r2_score', 0):.4f}")
            
            print("\n" + "─"*70)
            print("📉 MÉTRICAS DE ERROR")
            print("─"*70)
            print(f"  • RMSE (Validación):      {metrics.get('rmse_val', 0):.4f}")
            print(f"  • RMSE (Entrenamiento):   {metrics.get('rmse_train', 0):.4f}")
            print(f"  • MAE (Validación):       {metrics.get('mae_val', 0):.4f}")
            print(f"  • MSE (Validación):       {metrics.get('mse_val', 0):.4f}")
            
            print("\n" + "─"*70)
            print("🎯 MÉTRICAS DE RANKING (NDCG)")
            print("─"*70)
            print(f"  ★ NDCG General:           {metrics.get('ndcg', 0):.2f}%")
            print(f"  ★ NDCG@5:                 {metrics.get('ndcg_k5', 0):.2f}%")
            print(f"  ★ NDCG@10:                {metrics.get('ndcg_k10', 0):.2f}%")
            
            error_dist = metrics.get('error_distribution', {})
            print("\n" + "─"*70)
            print("📊 DISTRIBUCIÓN DE ERRORES")
            print("─"*70)
            print(f"  • Predicciones exactas:   {error_dist.get('exact_predictions_pct', 0):.2f}%")
            print(f"  • Error ±1:               {error_dist.get('off_by_1_pct', 0):.2f}%")
            print(f"  • Error ≥2:               {error_dist.get('off_by_2_plus_pct', 0):.2f}%")
            
            print("\n" + "─"*70)
            print("📦 DATOS DE ENTRENAMIENTO")
            print("─"*70)
            print(f"  • Muestras entrenamiento: {metrics.get('train_samples', 0):,}")
            print(f"  • Muestras validación:    {metrics.get('val_samples', 0):,}")
            
            pred_range = metrics.get('prediction_range', {})
            print("\n" + "─"*70)
            print("🎲 RANGO DE PREDICCIONES")
            print("─"*70)
            print(f"  • Mínimo:                 {pred_range.get('min', 0):.4f}")
            print(f"  • Máximo:                 {pred_range.get('max', 0):.4f}")
            print(f"  • Media:                  {pred_range.get('mean', 0):.4f}")
            
            print("\n" + "═"*70)
            print("✅ Métricas obtenidas exitosamente")
            print("═"*70)
        else:
            print("❌ No hay modelos disponibles")
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)
        
except Exception as e:
    print(f"❌ Error obteniendo métricas: {str(e)}")

## 📈 PASO 3: Visualizaciones del Modelo

Generamos 6 gráficas clave para analizar el rendimiento del modelo

In [ ]:
# Configuración general de gráficas
fig = plt.figure(figsize=(20, 12))
fig.suptitle('📊 ANÁLISIS COMPLETO DEL MODELO ML - SISTEMA DE RECOMENDACIÓN NUTRICIONAL', 
             fontsize=20, fontweight='bold', y=0.995)

# Colores personalizados
colors_accuracy = ['#2ecc71', '#3498db']
colors_error = ['#e74c3c', '#e67e22', '#f39c12', '#9b59b6']
colors_ndcg = ['#1abc9c', '#16a085', '#27ae60']
colors_dist = ['#2ecc71', '#f39c12', '#e74c3c']

### Gráfica 1: Accuracy Exacto vs ±1

In [ ]:
# Gráfica 1: Comparación de Accuracy
ax1 = plt.subplot(2, 3, 1)
accuracy_data = [
    metrics.get('accuracy_exact', 0),
    metrics.get('accuracy_tolerance_1', 0)
]
accuracy_labels = ['Exacto', '±1 punto']

bars1 = ax1.bar(accuracy_labels, accuracy_data, color=colors_accuracy, edgecolor='black', linewidth=2)
ax1.set_ylabel('Porcentaje (%)', fontsize=12, fontweight='bold')
ax1.set_title('🎯 Accuracy del Modelo', fontsize=14, fontweight='bold', pad=20)
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores en las barras
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Caja de información
info_text1 = f"✓ {accuracy_data[0]:.1f}% predicciones perfectas\n✓ {accuracy_data[1]:.1f}% con error ≤1"
ax1.text(0.5, 0.95, info_text1, transform=ax1.transAxes,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
         verticalalignment='top', horizontalalignment='center', fontsize=10)

### Gráfica 2: Métricas de Error

In [ ]:
# Gráfica 2: Métricas de Error
ax2 = plt.subplot(2, 3, 2)
error_metrics = [
    metrics.get('rmse_val', 0),
    metrics.get('rmse_train', 0),
    metrics.get('mae_val', 0),
    metrics.get('mse_val', 0)
]
error_labels = ['RMSE\nVal', 'RMSE\nTrain', 'MAE\nVal', 'MSE\nVal']

bars2 = ax2.bar(error_labels, error_metrics, color=colors_error, edgecolor='black', linewidth=2)
ax2.set_ylabel('Valor de Error', fontsize=12, fontweight='bold')
ax2.set_title('📉 Métricas de Error', fontsize=14, fontweight='bold', pad=20)
ax2.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Caja de información
info_text2 = f"RMSE: Error cuadrático medio\nMAE: Error absoluto medio\nMenor es mejor"
ax2.text(0.5, 0.95, info_text2, transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3),
         verticalalignment='top', horizontalalignment='center', fontsize=9)

### Gráfica 3: NDCG Scores (Ranking)

In [ ]:
# Gráfica 3: NDCG Scores
ax3 = plt.subplot(2, 3, 3)
ndcg_data = [
    metrics.get('ndcg', 0),
    metrics.get('ndcg_k5', 0),
    metrics.get('ndcg_k10', 0)
]
ndcg_labels = ['NDCG\nGeneral', 'NDCG@5', 'NDCG@10']

bars3 = ax3.bar(ndcg_labels, ndcg_data, color=colors_ndcg, edgecolor='black', linewidth=2)
ax3.set_ylabel('Porcentaje (%)', fontsize=12, fontweight='bold')
ax3.set_title('🎯 NDCG - Calidad del Ranking', fontsize=14, fontweight='bold', pad=20)
ax3.set_ylim(0, 100)
ax3.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores
for bar in bars3:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Caja de información
avg_ndcg = np.mean(ndcg_data)
info_text3 = f"⭐ Promedio: {avg_ndcg:.1f}%\nMide qué tan bien\nordena las recomendaciones"
ax3.text(0.5, 0.95, info_text3, transform=ax3.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.4),
         verticalalignment='top', horizontalalignment='center', fontsize=10)

### Gráfica 4: Distribución de Errores (Pie Chart)

In [ ]:
# Gráfica 4: Distribución de Errores
ax4 = plt.subplot(2, 3, 4)
error_dist = metrics.get('error_distribution', {})
dist_data = [
    error_dist.get('exact_predictions_pct', 0),
    error_dist.get('off_by_1_pct', 0),
    error_dist.get('off_by_2_plus_pct', 0)
]
dist_labels = ['Exactas', 'Error ±1', 'Error ≥2']

wedges, texts, autotexts = ax4.pie(dist_data, labels=dist_labels, autopct='%1.1f%%',
                                     colors=colors_dist, startangle=90,
                                     explode=(0.05, 0, 0),
                                     shadow=True, textprops={'fontweight': 'bold'})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(11)

ax4.set_title('📊 Distribución de Errores de Predicción', fontsize=14, fontweight='bold', pad=20)

# Leyenda con información
legend_text = [f'{label}: {value:.1f}%' for label, value in zip(dist_labels, dist_data)]
ax4.legend(legend_text, loc='upper left', bbox_to_anchor=(0.85, 1), fontsize=10)

### Gráfica 5: Muestras de Entrenamiento vs Validación

In [ ]:
# Gráfica 5: Muestras Train vs Validation
ax5 = plt.subplot(2, 3, 5)
samples_data = [
    metrics.get('train_samples', 0),
    metrics.get('val_samples', 0)
]
samples_labels = ['Entrenamiento', 'Validación']
sample_colors = ['#3498db', '#e67e22']

bars5 = ax5.bar(samples_labels, samples_data, color=sample_colors, edgecolor='black', linewidth=2)
ax5.set_ylabel('Cantidad de Muestras', fontsize=12, fontweight='bold')
ax5.set_title('📦 Dataset: Entrenamiento vs Validación', fontsize=14, fontweight='bold', pad=20)
ax5.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores
for bar in bars5:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Caja de información
total_samples = sum(samples_data)
val_pct = (samples_data[1] / total_samples * 100) if total_samples > 0 else 0
info_text5 = f"Total: {total_samples:,} muestras\nValidación: {val_pct:.1f}%"
ax5.text(0.5, 0.95, info_text5, transform=ax5.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.4),
         verticalalignment='top', horizontalalignment='center', fontsize=10)

### Gráfica 6: Rango de Predicciones

In [ ]:
# Gráfica 6: Rango de Predicciones
ax6 = plt.subplot(2, 3, 6)
pred_range = metrics.get('prediction_range', {})
range_data = [
    pred_range.get('min', 0),
    pred_range.get('mean', 0),
    pred_range.get('max', 0)
]
range_labels = ['Mínimo', 'Media', 'Máximo']
range_colors = ['#e74c3c', '#f39c12', '#2ecc71']

bars6 = ax6.bar(range_labels, range_data, color=range_colors, edgecolor='black', linewidth=2)
ax6.set_ylabel('Valor de Predicción', fontsize=12, fontweight='bold')
ax6.set_title('🎲 Rango de Predicciones del Modelo', fontsize=14, fontweight='bold', pad=20)
ax6.set_ylim(0, 5)
ax6.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores
for bar in bars6:
    height = bar.get_height()
    ax6.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Caja de información
pred_std = pred_range.get('max', 0) - pred_range.get('min', 0)
info_text6 = f"Rango: {pred_std:.2f}\nEscala: 1-5 puntos\n(preferencias)"
ax6.text(0.5, 0.95, info_text6, transform=ax6.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5),
         verticalalignment='top', horizontalalignment='center', fontsize=10)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

print("\n✅ Todas las gráficas generadas exitosamente")

## 📋 Resumen Final

In [ ]:
# Resumen final en formato tabla
print("\n" + "="*80)
print("📋 RESUMEN EJECUTIVO DEL MODELO".center(80))
print("="*80 + "\n")

# Crear DataFrame con métricas clave
summary_data = {
    'Métrica': [
        'Accuracy Exacto',
        'Accuracy ±1',
        'NDCG General',
        'NDCG@5',
        'RMSE Validación',
        'R² Score',
        'Muestras Totales'
    ],
    'Valor': [
        f"{metrics.get('accuracy_exact', 0):.2f}%",
        f"{metrics.get('accuracy_tolerance_1', 0):.2f}%",
        f"{metrics.get('ndcg', 0):.2f}%",
        f"{metrics.get('ndcg_k5', 0):.2f}%",
        f"{metrics.get('rmse_val', 0):.4f}",
        f"{metrics.get('r2_score', 0):.4f}",
        f"{metrics.get('train_samples', 0) + metrics.get('val_samples', 0):,}"
    ],
    'Interpretación': [
        '🎯 Predicciones perfectas',
        '✅ Predicciones aceptables',
        '⭐ Calidad del ranking',
        '🏆 Top 5 recomendaciones',
        '📉 Error de predicción',
        '📊 Varianza explicada',
        '📦 Datos de entrenamiento'
    ]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n" + "="*80)
print("💡 CONCLUSIONES:")
print("="*80)

# Análisis automático
ndcg_score = metrics.get('ndcg', 0)
accuracy = metrics.get('accuracy_exact', 0)

if ndcg_score >= 90:
    print("  ✅ EXCELENTE: El modelo tiene un rendimiento excepcional en ranking")
elif ndcg_score >= 85:
    print("  ✅ MUY BUENO: El modelo tiene un rendimiento sólido")
elif ndcg_score >= 80:
    print("  ⚠️  BUENO: El modelo funciona bien pero tiene margen de mejora")
else:
    print("  ⚠️  MEJORABLE: Considerar reentrenar con más datos o ajustar hiperparámetros")

if accuracy >= 35:
    print("  ✅ Alta precisión en predicciones exactas")
elif accuracy >= 25:
    print("  ✅ Precisión aceptable en predicciones exactas")
else:
    print("  ⚠️  Precisión baja, pero compensada con accuracy ±1")

print("\n  📌 El modelo está listo para producción" if ndcg_score >= 85 else "  ⚠️  Recomendado reentrenar antes de producción")

print("\n" + "="*80)
print("🎉 ANÁLISIS COMPLETADO".center(80))
print("="*80)

## 📊 BONUS: Comparación con Modelos Anteriores (Opcional)

In [ ]:
# Obtener todos los modelos para comparación
print("📊 Comparando con modelos anteriores...\n")

try:
    response = requests.get(status_url, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        
        if data['success']:
            models = sorted(data['models'], key=lambda x: x['created_at'])
            
            # Crear DataFrame comparativo
            comparison_data = []
            for i, model in enumerate(models, 1):
                m = model['metrics']
                comparison_data.append({
                    'Modelo': f"#{i}",
                    'Fecha': model['created_at'][11:16],
                    'Acc Exacto': f"{m.get('accuracy_exact', 0):.2f}%",
                    'NDCG': f"{m.get('ndcg', 0):.2f}%",
                    'RMSE': f"{m.get('rmse_val', 0):.4f}"
                })
            
            df_comparison = pd.DataFrame(comparison_data)
            print(df_comparison.to_string(index=False))
            
            # Encontrar el mejor
            best_model = max(models, key=lambda x: x['metrics'].get('ndcg', 0))
            best_index = models.index(best_model) + 1
            best_ndcg = best_model['metrics'].get('ndcg', 0)
            
            print(f"\n🏆 MEJOR MODELO: #{best_index} con NDCG = {best_ndcg:.2f}%")
            
except Exception as e:
    print(f"❌ Error en comparación: {str(e)}")